<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 16 · Unsupervised Learning and Representation

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Notebook Goals

This notebook makes Chapter 16 concrete by:

- building a cross-sectional feature panel from the price data,
- clustering assets into groups via k-means,
- visualizing clusters in PCA space, and
- summarizing cluster-level statistics for portfolio design.

### Getting Help While Using Unsupervised Methods

- Chapter 7 reminds you how we engineer features.
- Appendix B is the quick reference for pandas reshaping.
- Appendix C lists scikit-learn clustering and PCA utilities.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

### Data Loading
We load the cleaned price panel once for this notebook.

In [ ]:
prices = pd.read_csv(DATA_PATH,
parse_dates=["Date"]).set_index("Date").sort_index().ffill()

## 1. Feature Panel for Clustering

We build two features per asset: a 60-day return (momentum) and a 60-day rolling volatility. The latest row becomes our cross-sectional snapshot.

In [ ]:
prices = pd.read_csv(DATA_PATH,
parse_dates=["Date"]).set_index("Date").sort_index().ffill()
log_rets = np.log(prices / prices.shift(1)).dropna()
window = 60
feature_panel = pd.concat(
    {"momentum": prices.pct_change(window),
     "volatility": log_rets.rolling(window).std()},
    axis=1,
).dropna()
latest = feature_panel.iloc[-1].unstack(0)
scaler = StandardScaler()
X = scaler.fit_transform(latest)
assets = latest.index
latest.head()

## 2. K-Means Clustering

We group assets into a small number of clusters based on their scaled momentum and volatility features.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=0)
labels = kmeans.fit_predict(X)
clusters = pd.Series(labels, index=assets, name="cluster")
clusters

### 2.1 Silhouette Score

The silhouette score offers a quick sense of how well-separated the clusters are in feature space.

In [ ]:
silhouette = silhouette_score(X, labels)
silhouette

## 3. PCA Visualization

Principal Components Analysis (PCA) gives a low-dimensional view of the feature space that is easy to plot.

In [ ]:
pca = PCA(n_components=2)
components = pca.fit_transform(X)
fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(
    components[:, 0],
    components[:, 1],
    c=labels,
    cmap="coolwarm",
    s=160,
)
for idx, name in enumerate(assets):
    ax.annotate(
        name,
        (components[idx, 0], components[idx, 1]),
        textcoords="offset points",
        xytext=(6, 6),
    )
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Clusters in PCA Space")
plt.show()

## 4. Cluster-Level Summary

Cluster summaries help you understand how average momentum and volatility differ across groups.

In [ ]:
cluster_summary = latest.assign(cluster=labels).groupby("cluster").agg(["mean",
"std"])
cluster_summary

## 5. Exercises

### Exercise 1 – Hierarchical Clustering
Run an agglomerative clustering algorithm and compare labels to k-means.
<details><summary>Hint</summary>Use sklearn.cluster.AgglomerativeClustering and adjusted Rand index.</details>

### Exercise 2 – Time-Varying Clusters
Apply k-means on multiple historical dates to see how memberships change.
<details><summary>Hint</summary>Loop over selected dates, recompute features, and track label changes per asset.</details>

### Exercise 3 – Autoencoder Compression
Replace PCA with a small autoencoder and compare the latent factors.
<details><summary>Hint</summary>Reuse the Chapter 12 autoencoder, but train it on the latest feature matrix instead of time-series data.</details>

## 6. Takeaways

Clustering and PCA provide quick insight into how assets group together, which is useful for constraints, monitoring, and brainstorming new portfolio structures.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">